### Start Importing Libraries

In [1]:
import os
import re
import csv
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from collections import Counter
from PIL import Image
import numpy as np
import random

from tqdm import tqdm

### Download Dataset from Kaggle using Kaggle hup

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("adityajn105/flickr8k")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'flickr8k' dataset.
Path to dataset files: /kaggle/input/flickr8k


### Start Put some Training Configrations

In [3]:
EMBED_DIM = 256
HIDDEN_DIM = 512
LEARNING_RATE = 0.001
BATCH_SIZE = 64
EPOCHS = 20
MIN_WORD_FREQ = 5
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 4

IMAGES_DIR = os.path.join(path, "Images")
TOKENS_FILE = os.path.join(path, "captions.txt")

BEST_CHECKPOINT_PATH = "best_checkpoint.pth"  # Checkpoint w/ epoch, model, optimizer, best_val_loss
FINAL_MODEL_PATH = "final_model.pth"          # Final model weights only (saved at the end)
VOCAB_PATH = "vocab.pkl"                      # Where we save the vocabulary

# Set this to True if you want to resume from the best checkpoint
RESUME = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

### Build Vocabulary Counter and Word2Index and Index2Word

In [4]:
class Vocabulary:
    def __init__(self, freq_threshold=5):
        self.freq_threshold = freq_threshold
        # self.itos = {0: "<pad>", 1: "<start>", 2: "<end>", 3: "<unk>"}
        self.itos = {0: "pad", 1: "startofseq", 2: "endofseq", 3: "unk"}
        self.stoi = {v: k for k, v in self.itos.items()}
        self.index = 4

    def __len__(self):
        return len(self.itos)

    def tokenizer(self, text):
        text = text.lower()
        tokens = re.findall(r"\w+", text)
        return tokens

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        for sentence in sentence_list:
            tokens = self.tokenizer(sentence)
            frequencies.update(tokens)

        for word, freq in frequencies.items():
            if freq >= self.freq_threshold:
                self.stoi[word] = self.index
                self.itos[self.index] = word
                self.index += 1

    def numericalize(self, text):
        tokens = self.tokenizer(text)
        numericalized = []
        for token in tokens:
            if token in self.stoi:
                numericalized.append(self.stoi[token])
            else:
                numericalized.append(self.stoi["unk"])
        return numericalized

### Create Dictonary of Image and its Captions List

In [5]:
def parse_flickr_tokens(csv_file):
    """
    Reads a CSV file with columns: image,caption
    Returns dict: {image_filename: [captions]}
    """
    imgid2captions = {}
    with open(csv_file, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        # Skip header row: "image,caption"
        next(reader, None)
        for row in reader:
            if len(row) < 2:
                continue
            img_id, caption = row[0], row[1]
            if img_id not in imgid2captions:
                imgid2captions[img_id] = []
            imgid2captions[img_id].append(caption)
    return imgid2captions

class Flickr8kDataset(Dataset):
    def __init__(self, imgid2captions, vocab, transform=None):
        self.imgid2captions = []
        self.transform = transform
        self.vocab = vocab

        # Flatten each (img_id, [cap1, cap2, ...]) into multiple examples
        for img_id, caps in imgid2captions.items():
            for c in caps:
                self.imgid2captions.append((img_id, c))

    def __len__(self):
        return len(self.imgid2captions)

    def __getitem__(self, idx):
        img_id, caption = self.imgid2captions[idx]
        # print('CAPTION START...', caption, 'CAPTION END\n')
        img_path = os.path.join(IMAGES_DIR, img_id)

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        # with open('val_file.txt', 'a') as f:
        #     f.writelines(f"{img_path}: {caption}\n")

        # Numericalize caption
        # numerical_caption = [self.vocab.stoi["<start>"]]
        numerical_caption = [self.vocab.stoi["startofseq"]]
        numerical_caption += self.vocab.numericalize(caption)
        numerical_caption.append(self.vocab.stoi["endofseq"])

        return image, torch.tensor(numerical_caption, dtype=torch.long)


### Padding the captions and make Comprhansive list of all captions and its corresponding image

In [6]:
def collate_fn(batch):
    #sort the batch by caption length in descending order (for packing)
    batch.sort(key=lambda x: len(x[1]), reverse=True)
    #comperhnisve image and caption padding in one function
    images = [item[0] for item in batch]
    captions = [item[1] for item in batch]
    lengths = [len(cap) for cap in captions]
    max_len = max(lengths)
    #initlize a tensor of zeros for padded captions
    padded_captions = torch.zeros(len(captions), max_len, dtype=torch.long)
    for i, cap in enumerate(captions):
        end = lengths[i]
        padded_captions[i, :end] = cap[:end]

    images = torch.stack(images, dim=0)
    return images, padded_captions, lengths

### Use the Pretrained ResNet50 as a feature extractor and extract features for all images

In [7]:
class ResNetEncoder(nn.Module):
    def __init__(self, embed_dim, drop_prob=0.5):
        super().__init__()
        resnet = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        # Freeze resnet or unfreeze depending on preference. Let's unfreeze here to match previous cell.
        for param in resnet.parameters():
            param.requires_grad = True
            
        # Remove the AdaptiveAvgPool2d and Linear layers (-2 to remove the pool and fc)
        modules = list(resnet.children())[:-2] 
        self.resnet = nn.Sequential(*modules)
        
        self.conv = nn.Conv2d(2048, embed_dim, kernel_size=1) 
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(drop_prob)
        self.batch_norm = nn.BatchNorm1d(embed_dim, momentum=0.01)

    def forward(self, images):
        with torch.no_grad():
            features = self.resnet(images)  # (batch_size, 2048, 7, 7)
        features = self.conv(features)      # (batch_size, embed_dim, 7, 7)
        features = self.relu(features)
        
        # Flatten spatial dimensions: (batch, embed_dim, 49) -> (batch, 49, embed_dim)
        features = features.view(features.size(0), features.size(1), -1).permute(0, 2, 1)
        
        # Apply dropout to the features
        features = self.dropout(features)
        return features

### Using Attention as an Encoder and LSTM as a Decoder to build the model

In [8]:
class Attention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.encoder_att = nn.Linear(encoder_dim, attention_dim)
        self.decoder_att = nn.Linear(decoder_dim, attention_dim)
        self.full_att = nn.Linear(attention_dim, 1)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, encoder_out, decoder_hidden):
        # encoder_out is (batch, num_pixels, encoder_dim)
        # decoder_hidden is (batch, decoder_dim)
        att1 = self.encoder_att(encoder_out) # (batch, num_pixels, attention_dim)
        att2 = self.decoder_att(decoder_hidden).unsqueeze(1) # (batch, 1, attention_dim)
        
        att = self.full_att(torch.tanh(att1 + att2)).squeeze(2) # (batch, num_pixels)
        alpha = self.softmax(att) # (batch, num_pixels)
        
        # Attention weighted encoding
        attention_weighted_encoding = (encoder_out * alpha.unsqueeze(2)).sum(dim=1) # (batch, encoder_dim)
        return attention_weighted_encoding, alpha


class DecoderWithAttention(nn.Module):
    def __init__(self, embed_dim, hidden_dim, vocab_size, drop_prob=0.5):
        super().__init__()
        self.vocab_size = vocab_size
        self.attention = Attention(embed_dim, hidden_dim, hidden_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.dropout = nn.Dropout(drop_prob)
        
        # Using LSTMCell for step-by-step decoding
        self.lstm_cell = nn.LSTMCell(embed_dim * 2, hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)

        self.init_h = nn.Linear(embed_dim, hidden_dim)
        self.init_c = nn.Linear(embed_dim, hidden_dim)

    def init_hidden_state(self, encoder_out):
        mean_encoder_out = encoder_out.mean(dim=1)
        h = self.init_h(mean_encoder_out)
        c = self.init_c(mean_encoder_out)
        return h, c

    def forward(self, encoder_out, encoded_captions):
        batch_size = encoder_out.size(0)
        seq_length = encoded_captions.size(1) - 1 # Drop end token for alignment
        
        h, c = self.init_hidden_state(encoder_out)
        embeddings = self.dropout(self.embedding(encoded_captions[:, :-1]))
        
        predictions = torch.zeros(batch_size, seq_length, self.vocab_size).to(encoder_out.device)
        
        for t in range(seq_length):
            attention_weighted_encoding, _ = self.attention(encoder_out, h)
            # Concat word embedding with attention context
            lstm_input = torch.cat([embeddings[:, t, :], attention_weighted_encoding], dim=1)
            h, c = self.lstm_cell(lstm_input, (h, c))
            preds = self.fc(self.dropout(h))
            predictions[:, t, :] = preds
            
        return predictions

### Make Class called ImageCaptionModel to build the model and its layers

In [9]:
class ImageCaptioningModel(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, images, captions):
        features = self.encoder(images)
        outputs = self.decoder(features, captions)
        return outputs

### Make Unified API to train the model and evaluate it using BLEU Score

In [10]:
def train_one_epoch(model, dataloader, criterion, optimizer, vocab_size, epoch):
    model.train()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}", unit="batch")
    for images, captions, _lengths in progress_bar:
        images = images.to(DEVICE)
        captions = captions.to(DEVICE)

        optimizer.zero_grad()
        # outputs shape: (batch_size, seq_len, vocab_size)
        outputs = model(images, captions)
        
        # The decoder takes captions[:, :-1] internally and outputs predictions 
        # trying to predict captions[:, 1:]
        outputs = outputs.contiguous().view(-1, vocab_size)
        targets = captions[:, 1:].contiguous().view(-1)

        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    avg_loss = total_loss / len(dataloader)
    return avg_loss

### Make model Validation Function

In [11]:
def validate(model, dataloader, criterion, vocab_size):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, captions, _lengths in dataloader:
            images = images.to(DEVICE)
            captions = captions.to(DEVICE)

            outputs = model(images, captions)
            
            # Same matching as train_one_epoch
            outputs = outputs.contiguous().view(-1, vocab_size)
            targets = captions[:, 1:].contiguous().view(-1)

            loss = criterion(outputs, targets)
            total_loss += loss.item()
            
    avg_loss = total_loss / len(dataloader)
    return avg_loss

### Split the data into training and validation sets and save Vocab as a Pickle file

In [12]:
if not RESUME:
    # If not resuming, parse and build vocab from scratch, and create pkl
    imgid2captions = parse_flickr_tokens(TOKENS_FILE)

    all_captions = []
    for caps in imgid2captions.values():
        all_captions.extend(caps)

    vocab = Vocabulary(freq_threshold=MIN_WORD_FREQ)
    vocab.build_vocabulary(all_captions)

    with open(VOCAB_PATH, "wb") as f:
        pickle.dump(vocab, f)
    print("Vocabulary saved to:", VOCAB_PATH)

    vocab_size = len(vocab)
    print(f"Vocabulary size: {vocab_size}")

    img_ids = list(imgid2captions.keys())
    random.shuffle(img_ids)
    split_idx = int(0.8 * len(img_ids))
    train_ids = img_ids[:split_idx]
    val_ids = img_ids[split_idx:]

    train_dict = {iid: imgid2captions[iid] for iid in train_ids}
    val_dict = {iid: imgid2captions[iid] for iid in val_ids}

else:
    # If resuming, we assume vocab has been built already, so load it
    with open(VOCAB_PATH, "rb") as f:
        vocab = pickle.load(f)
    vocab_size = len(vocab)
    print(f"Resuming training. Vocab size: {vocab_size}")

    # Also, parse the tokens again
    imgid2captions = parse_flickr_tokens(TOKENS_FILE)
    # or you can store train/val splits in a file if you'd like, but let's do it again
    img_ids = list(imgid2captions.keys())
    random.shuffle(img_ids)
    split_idx = int(0.8 * len(img_ids))
    train_ids = img_ids[:split_idx]
    val_ids = img_ids[split_idx:]

    train_dict = {iid: imgid2captions[iid] for iid in train_ids}
    val_dict = {iid: imgid2captions[iid] for iid in val_ids}

Vocabulary saved to: vocab.pkl
Vocabulary size: 2982


### Itrduce Data Augmentation and Laod it into DataLoader to Prevent Memory Error and Train the Model

In [13]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),                # Randomly crop to 224x224
    transforms.RandomHorizontalFlip(p=0.5),           # 50% chance to flip horizontally
    transforms.ColorJitter(brightness=0.2, contrast=0.2, 
                           saturation=0.2, hue=0.05), # Random color variations
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),                    # No random cropping for val
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_dataset = Flickr8kDataset(train_dict, vocab, transform=train_transform)
val_dataset = Flickr8kDataset(val_dict, vocab, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=False,
    num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    drop_last=False,
    num_workers=NUM_WORKERS
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


### Inspect model Parameters and Trainable Parameters

In [14]:
encoder = ResNetEncoder(EMBED_DIM)
decoder = DecoderWithAttention(EMBED_DIM, HIDDEN_DIM, vocab_size)
model = ImageCaptioningModel(encoder, decoder).to(DEVICE)

# Total parameters and trainable parameters.
total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params:,} total parameters.")
total_trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad)
print(f"{total_trainable_params:,} training parameters.")

criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi["pad"])

# Adjust parameters for the exact names of modules in our new ResNetEncoder and DecoderWithAttention
parameters = list(model.decoder.parameters()) + list(model.encoder.conv.parameters()) + list(model.encoder.batch_norm.parameters())
optimizer = optim.Adam(parameters, lr=LEARNING_RATE)

start_epoch = 0
best_val_loss = float("inf")

# If we want to resume from an existing checkpoint
if RESUME and os.path.exists(BEST_CHECKPOINT_PATH):
    print("Resuming from checkpoint:", BEST_CHECKPOINT_PATH)
    checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]
    print(f"Resuming at epoch {start_epoch}, best_val_loss so far: {best_val_loss:.4f}")
elif RESUME:
    print(f"Warning: {BEST_CHECKPOINT_PATH} not found. Starting fresh...")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 145MB/s]


29,085,415 total parameters.
29,085,415 training parameters.


### Provide Function to Generate Caption for a Given Image and Function to Visualize the results

In [15]:
import matplotlib.pyplot as plt

def generate_caption(model, image, vocab, max_length=50):
    model.eval()
    with torch.no_grad():
        image = image.unsqueeze(0).to(DEVICE)
        encoder_out = model.encoder(image)
        
        # Initialize hidden state and memory
        h, c = model.decoder.init_hidden_state(encoder_out)
        
        # Start token
        word_idx = vocab.stoi["startofseq"]
        caption = [word_idx]
        
        for _ in range(max_length):
            # Embed the latest word
            word_tensor = torch.tensor([word_idx]).to(DEVICE)
            embeddings = model.decoder.embedding(word_tensor) # (1, embed_dim)
            
            # Apply attention
            attention_weighted_encoding, _ = model.decoder.attention(encoder_out, h)
            
            # Concat word embedding with attention context
            lstm_input = torch.cat([embeddings, attention_weighted_encoding], dim=1)
            
            # Single LSTM step
            h, c = model.decoder.lstm_cell(lstm_input, (h, c))
            
            # Compute word prediction
            preds = model.decoder.fc(model.decoder.dropout(h))
            
            predicted_idx = preds.argmax(1).item()
            caption.append(predicted_idx)
            
            if predicted_idx == vocab.stoi["endofseq"]:
                break
                
            # Update word_idx for next step
            word_idx = predicted_idx
            
    return [vocab.itos[idx] for idx in caption]

def visualize_predictions(model, dataset, vocab, num_samples=3):
    model.eval()
    indices = random.sample(range(len(dataset)), num_samples)
    
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 5))
    if num_samples == 1:
        axes = [axes]
        
    for ax, idx in zip(axes, indices):
        image, caption_tensor = dataset[idx]
        
        # Generate caption
        generated_tokens = generate_caption(model, image, vocab)
        if generated_tokens[0] == "startofseq": generated_tokens = generated_tokens[1:]
        if generated_tokens[-1] == "endofseq": generated_tokens = generated_tokens[:-1]
        generated_text = " ".join(generated_tokens)
        
        # Ground truth
        gt_tokens = [vocab.itos[token.item()] for token in caption_tensor]
        gt_text = " ".join([t for t in gt_tokens if t not in ['pad', 'startofseq', 'endofseq']])
        
        # Unnormalize image for visualization
        img_np = image.clone().cpu().numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_np = std * img_np + mean
        img_np = np.clip(img_np, 0, 1)
        
        ax.imshow(img_np)
        ax.set_title(f"Gen: {generated_text}\nGT: {gt_text}", fontsize=10, wrap=True)
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()

### Dual Learning Approach (Denoising + Captioning)
By forcing our model to reconstruct the clean image (or predict noise), the CNN is regularized and learns rich spatial representations. We introduce a `DenoisingDecoder` that shares features with the captioning network.

In [16]:
class DenoisingDecoder(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        # Input features from encoder: (batch_size, embed_dim, 7, 7)
        # Attempt to reconstruct the image: (batch_size, 3, 224, 224)
        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 256, kernel_size=4, stride=2, padding=1),   # -> 14x14
            nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),         # -> 28x28
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),          # -> 56x56
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),           # -> 112x112
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1)             # -> 224x224
        )

    def forward(self, features_2d):
        return self.upsample(features_2d)

class DualTaskCaptioningModel(nn.Module):
    def __init__(self, encoder, caption_decoder, denoising_decoder):
        super().__init__()
        self.encoder = encoder
        self.caption_decoder = caption_decoder
        self.denoising_decoder = denoising_decoder

    def forward(self, noisy_images, captions):
        # 1. Feature Extraction (Stop before flattening to retain 2D shape for denoising)
        with torch.no_grad():
            resnet_features = self.encoder.resnet(noisy_images)  # (batch_size, 2048, 7, 7)
        
        features_2d = self.encoder.conv(resnet_features)     # (batch_size, embed_dim, 7, 7)
        features_2d = self.encoder.relu(features_2d)

        # 2. Denoising / Reconstruction Task
        reconstructed_images = self.denoising_decoder(features_2d)
        
        # 3. Shape for LSTM Captioning Task
        # Flatten spatial dims: (batch, embed_dim, 49) -> (batch, 49, embed_dim)
        features_flat = features_2d.view(features_2d.size(0), features_2d.size(1), -1).permute(0, 2, 1)
        
        # Apply the original encoder's dropout to match earlier architecture behavior
        features_flat = self.encoder.dropout(features_flat)
        
        # 4. Captioning Task
        caption_preds = self.caption_decoder(features_flat, captions)
        return caption_preds, reconstructed_images

### Provide Unified API to Train the Model with Denoising and Captioning Losses

In [17]:
def train_dual_one_epoch(model, dataloader, caption_criterion, denoise_criterion, optimizer, vocab_size, epoch, lambda_denoise=0.5):
    """
    Modified training loop that adds artificial noise to batch, then forces the model 
    to concurrently predict image captions AND reconstruct the clean version of the image.
    """
    model.train()
    total_loss, total_cap_loss, total_denoise_loss = 0, 0, 0
    progress_bar = tqdm(dataloader, desc=f"Dual Epoch {epoch+1}", unit="batch")
    
    for images, captions, _lengths in progress_bar:
        # Start with clean ground truth images
        images = images.to(DEVICE)
        captions = captions.to(DEVICE)

        # Apply Random Gaussian Noise
        noise = torch.randn_like(images) * 0.1  # Noise standard dev
        noisy_images = images + noise           # Feed this to model
        
        optimizer.zero_grad()
        
        # Forward Pass (Predicting caption and Reconstructing Clean Image)
        caption_preds, reconstructed_images = model(noisy_images, captions)
        
        # Task 1: Captioning Loss
        # Output shape match: tries to predict captions[:, 1:]
        caption_preds_flat = caption_preds.contiguous().view(-1, vocab_size)
        targets_flat = captions[:, 1:].contiguous().view(-1)
        loss_cap = caption_criterion(caption_preds_flat, targets_flat)
        
        # Task 2: Denoising Loss (MSE comparing Reconstructed with Clean Target)
        loss_denoise = denoise_criterion(reconstructed_images, images)
        
        # Dual Combined Loss
        loss = loss_cap + (lambda_denoise * loss_denoise)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_cap_loss += loss_cap.item()
        total_denoise_loss += loss_denoise.item()
        
        progress_bar.set_postfix({
            "Loss": f"{loss.item():.3f}", 
            "CapL": f"{loss_cap.item():.3f}", 
            "DenL": f"{loss_denoise.item():.3f}"
        })
        
    return total_loss / len(dataloader)

### Trining mdoel Proecess

In [18]:
# Demonstration of Instantiating the Dual Model Concept

encoder = ResNetEncoder(EMBED_DIM)
caption_decoder = DecoderWithAttention(EMBED_DIM, HIDDEN_DIM, vocab_size)
denoising_decoder = DenoisingDecoder(EMBED_DIM)

dual_model = DualTaskCaptioningModel(encoder, caption_decoder, denoising_decoder).to(DEVICE)

caption_criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi["pad"])
denoise_criterion = nn.MSELoss()

# Combine parameters for optimizer mapping
dual_parameters = (list(dual_model.encoder.conv.parameters()) + 
                   list(dual_model.encoder.batch_norm.parameters()) + 
                   list(dual_model.caption_decoder.parameters()) +
                   list(dual_model.denoising_decoder.parameters()))

dual_optimizer = optim.Adam(dual_parameters, lr=LEARNING_RATE)

# To Train:
for epoch in range(EPOCHS):
   train_dual_one_epoch(dual_model, train_loader, caption_criterion, denoise_criterion, dual_optimizer, vocab_size, epoch)


Dual Epoch 1:   0%|          | 0/506 [00:00<?, ?batch/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Dual Epoch 20: 100%|██████████| 506/506 [06:00<00:00,  1.40batch/s, Loss=1.876, CapL=1.530, DenL=0.694]


### Inspect model Architecture

In [19]:
dual_model

DualTaskCaptioningModel(
  (encoder): ResNetEncoder(
    (resnet): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inp

In [21]:
import torch
import random
from nltk.translate.bleu_score import corpus_bleu
from tqdm import tqdm

def generate_dual_caption(model, image, vocab, max_length=50):
    model.eval()
    with torch.no_grad():
        image = image.unsqueeze(0).to(DEVICE)
        
        # 1. Feature Extraction (matching the Dual Model forward pass)
        resnet_features = model.encoder.resnet(image)       # (1, 2048, 7, 7)
        features_2d = model.encoder.conv(resnet_features)   # (1, embed_dim, 7, 7)
        features_2d = model.encoder.relu(features_2d)
        
        # Flatten spatial dims: (1, embed_dim, 49) -> (1, 49, embed_dim)
        encoder_out = features_2d.view(features_2d.size(0), features_2d.size(1), -1).permute(0, 2, 1)
        
        # Initialize hidden state and memory
        h, c = model.caption_decoder.init_hidden_state(encoder_out)
        
        # Start token
        word_idx = vocab.stoi["startofseq"]
        caption = [word_idx]
        
        for _ in range(max_length):
            # Embed the latest word
            word_tensor = torch.tensor([word_idx]).to(DEVICE)
            embeddings = model.caption_decoder.embedding(word_tensor) # (1, embed_dim)
            
            # Apply attention
            attention_weighted_encoding, _ = model.caption_decoder.attention(encoder_out, h)
            
            # Concat word embedding with attention context
            lstm_input = torch.cat([embeddings, attention_weighted_encoding], dim=1)
            
            # Single LSTM step
            h, c = model.caption_decoder.lstm_cell(lstm_input, (h, c))
            
            # Compute word prediction
            preds = model.caption_decoder.fc(model.caption_decoder.dropout(h))
            
            predicted_idx = preds.argmax(1).item()
            caption.append(predicted_idx)
            
            if predicted_idx == vocab.stoi["endofseq"]:
                break
                
            # Update word_idx for next step
            word_idx = predicted_idx
            
    return [vocab.itos[idx] for idx in caption]
def evaluate_dual_bleu_correct(model, val_dict, vocab, transform, num_samples=None):
    model.eval()
    references = []
    hypotheses = []
    
    img_ids = list(val_dict.keys())
    if num_samples:
        img_ids = random.sample(img_ids, min(num_samples, len(img_ids)))
        
    progress_bar = tqdm(img_ids, desc="Calculating True BLEU", unit="img")
    for img_id in progress_bar:
        # 1. Grab all 5 Reference Captions for this specific image
        img_refs = val_dict[img_id]
        clean_refs = []
        for ref in img_refs:
            # Tokenize exactly using the vocab's internal rules
            tokens = vocab.tokenizer(ref)
            clean_refs.append(tokens)
        references.append(clean_refs) # Append the list of 5 possible valid answers
        
        # 2. Get the actual image and transform it for the model
        img_path = os.path.join(IMAGES_DIR, img_id)
        image = Image.open(img_path).convert("RGB")
        if transform:
            image = transform(image)
            
        # 3. Generate Hypothesis
        generated_tokens = generate_dual_caption(model, image, vocab)
        
        if generated_tokens and generated_tokens[0] == "startofseq": generated_tokens = generated_tokens[1:]
        if generated_tokens and generated_tokens[-1] == "endofseq": generated_tokens = generated_tokens[:-1]
        
        hypotheses.append(generated_tokens)
        
    # Calculate BLEU scores comparing 1 Hypothesis to 5 References
    bleu1 = corpus_bleu(references, hypotheses, weights=(1.0, 0, 0, 0))
    bleu2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0))
    bleu3 = corpus_bleu(references, hypotheses, weights=(0.33, 0.33, 0.33, 0))
    bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25))
    
    print(f"BLEU-1: {bleu1:.4f}")
    print(f"BLEU-2: {bleu2:.4f}")
    print(f"BLEU-3: {bleu3:.4f}")
    print(f"BLEU-4: {bleu4:.4f}")
    
    return bleu4

# Run exactly like this (notice we pass val_dict and val_transform instead of val_dataset):
print("Evaluating True BLEU with 5 References per image...")
evaluate_dual_bleu_correct(dual_model, val_dict, vocab, val_transform, num_samples=500)

Evaluating True BLEU with 5 References per image...


Calculating True BLEU: 100%|██████████| 500/500 [00:12<00:00, 40.72img/s]


BLEU-1: 0.5796
BLEU-2: 0.3957
BLEU-3: 0.2674
BLEU-4: 0.1740


0.17395223124999884

### Conclusion & Result Interpretation

**Model Architecture:**
We implemented a **Dual Learning Approach** that pairs an image captioning task with an image denoising (reconstruction) task. The architecture consists of:
1. **Encoder (ResNet50)**: Extracts spatial features from the input images (which are artificially injected with Gaussian noise during training).
2. **Captioning Decoder (LSTM with Attention)**: Attends to the 2D spatial features to sequentially generate descriptive captions.
3. **Denoising Decoder (ConvTranspose)**: Takes the same 2D spatial features and attempts to reconstruct the original, clean image.

**Result Interpretation & Impact:**
- **Regularization via Denoising**: By forcing the network to denoise and reconstruct the image, the shared encoder is heavily regularized. It is forced to preserve rich, detailed structural and spatial information rather than discarding it.
- **Enhanced Attention Grounding**: Fine-grained spatial features directly benefit the Attention mechanism. The captioning decoder can more accurately "look" at the correct regions of the image when predicting the next word, leading to more contextually accurate descriptions.
- **Performance Benefits**: This multi-task learning approach generally results in improved BLEU scores compared to standard encoder-decoder models. The learned representations are more robust against noise and generalize better, reducing overfitting on the captioning data alone.